# Classification Track - Review 2 Part B

**Problem:** Predict whether a room is occupied (`Occupancy` = 0 or 1) from environmental sensors.

**Dataset:** UCI Occupancy Detection. This notebook continues `notebooks/classification.ipynb` (Review 1 Part A): same files, same CO2 filter, same features, and the same official train/test split.

**Part A, refit with the hyperparameters already chosen:** Logistic Regression, K-Nearest Neighbors, Gaussian Naive Bayes, Decision Tree, Support Vector Machine.

**Part B, trained here:** Random Forest, AdaBoost, Gradient Boosting, Bagging, MLP.

**Metrics for every model:** Accuracy, Precision (weighted), Recall (weighted), F1 (weighted), ROC-AUC, and a confusion matrix. The last table holds all ten classifiers.


In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    BaggingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

sns.set_theme(palette="colorblind")
RANDOM_STATE = 42


## 1. Same room-occupancy split as Part A

Training file: `data/datatraining.txt`. Test files: `data/datatest.txt` and `data/datatest2.txt`, concatenated. This is the UCI official split used in Review 1, so Part B is compared on the same rows as Part A.

CO2 readings above 2000 are removed as sensor anomalies, using the same rule as Part A. Features are Temperature, Light, CO2, HumidityRatio, seconds since midnight (NSM), Weekday, and Weekend. Raw Humidity is left out because HumidityRatio is already the temperature-adjusted humidity supplied with the dataset.


In [2]:
def clean_data(df):
    # Same anomaly rule as Review 1 Part A.
    return df.query("CO2 <= 2000").reset_index(drop=True)


def prepare_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["NSM"] = (
        df["date"].dt.hour * 3600
        + df["date"].dt.minute * 60
        + df["date"].dt.second
    )
    # Monday-Friday vs Saturday-Sunday. Weekend is the complement of Weekday.
    df["Weekday"] = df["date"].dt.dayofweek < 5
    df["Weekend"] = df["date"].dt.dayofweek >= 5
    return df


FEATURES = [
    "Temperature",
    "Light",
    "CO2",
    "HumidityRatio",
    "NSM",
    "Weekday",
    "Weekend",
]
TARGET = "Occupancy"

train = prepare_features(clean_data(pd.read_csv("../data/datatraining.txt")))
test = prepare_features(clean_data(pd.concat([
    pd.read_csv("../data/datatest.txt"),
    pd.read_csv("../data/datatest2.txt"),
], ignore_index=True)))

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("Train occupied rate: {:.4f}".format(y_train.mean()))
print("Test occupied rate: {:.4f}".format(y_test.mean()))
print("Missing values:", int(pd.concat([X_train, X_test]).isna().sum().sum()))


Training set: (8102, 7)
Testing set: (12414, 7)
Train occupied rate: 0.2083
Test occupied rate: 0.2434
Missing values: 0


### Split observation

**Observation:** After the CO2 filter, training has **8,102** rows and the official test files have **12,414**. Occupied rows are the minority: **20.83%** of training rows and **24.34%** of test rows. The seven model features have no missing values. The test file is larger than the training file because that is the published UCI split; Part B keeps it so every model is scored on the same rows as Part A.
